# Week 2 Day 5 — Modern Portfolio Theory

Use expected returns, covariance, and random long-only weights to build an efficient frontier and compare it to a benchmark.

## Objective

Construct a modern portfolio theory analysis using repo data and PortfolioSimulator.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from tradinglab.data_feed import DataFeed
from tradinglab.simulator import PortfolioSimulator

plt.style.use('seaborn-v0_8')

In [ ]:
repo_root = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / 'data' / 'egx').exists() and (candidate / 'src').exists():
        repo_root = candidate
        break
if repo_root is None:
    raise FileNotFoundError('Could not locate repository root')
feed = DataFeed.from_dir(repo_root / 'data' / 'egx')
returns = feed.returns[:, :]
mean_returns = returns.mean(axis=0)
cov_matrix = np.cov(returns.T)
print('mean returns shape', mean_returns.shape)
print('covariance shape', cov_matrix.shape)

In [ ]:
rng = np.random.default_rng(0)
weights = []
for _ in range(400):
    w = rng.random(len(feed.symbols))
    w = w / w.sum()
    weights.append(w)
weights = np.array(weights)
portfolio_returns = weights @ mean_returns
portfolio_vol = np.sqrt(np.einsum('ij,jk,ik->i', weights, cov_matrix, weights))
sharpes = portfolio_returns / np.where(portfolio_vol > 0, portfolio_vol, 1e-9)
frontier = pd.DataFrame({'return': portfolio_returns, 'risk': portfolio_vol, 'sharpe': sharpes})
frontier = frontier.sort_values('risk').reset_index(drop=True)
frontier.head()

In [ ]:
best_sharpe_idx = frontier['sharpe'].idxmax()
min_var_idx = frontier['risk'].idxmin()
max_sharpe_weights = weights[best_sharpe_idx]
min_var_weights = weights[min_var_idx]
print('max sharpe weights', np.round(max_sharpe_weights, 3))
print('min variance weights', np.round(min_var_weights, 3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(frontier['risk'], frontier['return'], c=frontier['sharpe'], cmap='viridis', alpha=0.6)
axes[0].set_xlabel('Risk')
axes[0].set_ylabel('Expected Return')
axes[0].set_title('Efficient Frontier')
axes[1].bar(feed.symbols, max_sharpe_weights)
axes[1].set_title('Maximum Sharpe Allocation')
axes[1].set_ylabel('Weight')
plt.tight_layout()
plt.show()

## Conclusion

The notebook uses expected returns and covariance to approximate the efficient frontier and chooses a maximum-Sharpe allocation. This can be compared against MLP and LSTM portfolios using PortfolioSimulator.